<a href="https://colab.research.google.com/github/alphonsetandeta-hub/Fusion-des-donnees/blob/main/FUSION_DES_DONNEES_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌱 Fanaye — Fusion des données (1 seule feuille)
Fusionne toutes les feuilles journalières de `Fanaye1.xls` en une seule feuille Excel,
avec les coordonnées GPS répétées tous les 24 points + colonnes **Variete** et **Site**.

Ce notebook est conçu pour fusionner les données de plusieurs feuilles d'un fichier Excel (`Fanaye1.xls`) en une seule feuille consolidée. Il extrait également les coordonnées GPS d'une feuille spécifique et les répète pour chaque bloc de 24 points de données. Enfin, il exporte les données nettoyées et fusionnées dans un nouveau fichier Excel (`fanaye1_grand_parc_fanaye.xlsx`).

## Étape 1 — Installation

In [ ]:
!pip install openpyxl -q
print('✅ Librairies prêtes')

✅ Librairies prêtes


Cette section installe la bibliothèque `openpyxl`, qui est nécessaire pour lire et écrire des fichiers Excel, en particulier pour les opérations de style et de formatage des cellules.

## Étape 2 — Upload du fichier `Fanaye1.xls`

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'✅ Fichier chargé : {filename}')

Saving Alioune1.xls to Alioune1.xls
✅ Fichier chargé : Alioune1.xls


Ici, nous téléchargeons le fichier Excel source (`Fanaye1.xls`) directement dans l'environnement Colab. Le nom du fichier est stocké dans la variable `filename` pour une utilisation ultérieure.

## Étape 3 — Lecture, fusion et export Excel

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ═══════════════════════════════════════════
#  PARAMÈTRES — modifiez si nécessaire
# ═══════════════════════════════════════════
COORD_SHEET = 'coordonne'
VARIETE     = 'alioune1'
SITE        = 'alioune'
NB_POINTS   = 24
OUTPUT      = 'alioune1_grand_parc_alioune.xlsx'
# ═══════════════════════════════════════════

# ── 1. Lecture de toutes les feuilles ──────
all_sheets = pd.read_excel(filename, sheet_name=None, header=None, engine='openpyxl')
print(f'📋 {len(all_sheets)} feuilles trouvées : {list(all_sheets.keys())}')

# ── 2. Coordonnées GPS ─────────────────────
# Based on the kernel state for 'Alioune1.xls', the 'coordonne' sheet seems to have:
# Column 0: Latitude
# Column 1: Longitude
# Other expected columns (Elevation, Time, Name/Point) are not present in the loaded DataFrame.
coord_raw  = all_sheets[COORD_SHEET]
coord_rows = []
for i in range(1, NB_POINTS + 1):
    r = coord_raw.iloc[i]
    coord_rows.append({
        'Point'    : f'F{i}', # Generate point name, e.g., F1, F2, ...
        'Latitude' : r[0],
        'Longitude': r[1],
        'Elevation': 0.0,     # Default to 0.0 if elevation data is not available
    })
print(f'📍 {len(coord_rows)} points GPS chargés')

# ── 3. Fusion de toutes les feuilles ───────
data_sheets = {k: v for k, v in all_sheets.items() if k != COORD_SHEET}
merged = []

for date, df in data_sheets.items():
    for i in range(1, len(df)):
        row = df.iloc[i]
        try:
            sn = int(row[0])
        except:
            sn = i
        coord = coord_rows[(sn - 1) % NB_POINTS]
        merged.append([
            date,               # Date
            sn,                 # SN
            row[1],             # Description
            row[2],             # Time
            row[3],             # Temp
            row[4],             # Humidity
            row[5],             # EC
            row[6],             # PH
            row[7],             # Nitrogen
            row[8],             # Phosphorus
            row[9],             # Potassium
            row[10],            # Salinity
            coord['Point'],     # Point GPS
            coord['Latitude'],  # Latitude
            coord['Longitude'], # Longitude
            coord['Elevation'], # Elevation (m)
            VARIETE,            # Variete
            SITE,               # site
        ])

print(f'✅ {len(merged)} lignes fusionnées  |  {len(data_sheets)} jours')

# ── 4. Création du fichier Excel ───────────
HEADERS = [
    'Date', 'SN', 'Description', 'Time',
    'Temp (℃)', 'Humidity (%)', 'EC (μS/cm)', 'PH',
    'Nitrogen (mg/kg)', 'Phosphorus (mg/kg)', 'Potassium (mg/kg)', 'Salinity (mg/kg)',
    'Point GPS', 'Latitude', 'Longitude', 'Elevation (m)',
    'Variete', 'site'
]
COL_WIDTHS = [14,5,14,22,10,12,12,7,16,16,16,14,10,12,12,14,12,10]

wb = Workbook()
ws = wb.active
ws.title = 'Données Fusionnées'

# En-têtes
hdr_font  = Font(bold=True, color='000000', name='Arial', size=10)
ctr       = Alignment(horizontal='center', vertical='center', wrap_text=True)

for col, h in enumerate(HEADERS, 1):
    c           = ws.cell(row=1, column=col, value=h)
    c.font      = hdr_font
    c.alignment = ctr
ws.row_dimensions[1].height = 35

# Données
fill_a   = PatternFill('solid', fgColor='FFFFFF')  # white
fill_b   = PatternFill('solid', fgColor='FFFFFF')  # white
data_ft  = Font(name='Arial', size=9)
data_al  = Alignment(horizontal='center', vertical='center')

for r_idx, row_data in enumerate(merged, 2):
    group    = (r_idx - 2) // NB_POINTS
    row_fill = fill_a if group % 2 == 0 else fill_b
    for c_idx, val in enumerate(row_data, 1):
        c           = ws.cell(row=r_idx, column=c_idx, value=val)
        c.font      = data_ft
        c.alignment = data_al
        c.fill      = row_fill

# Largeurs
for i, w in enumerate(COL_WIDTHS, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

ws.freeze_panes = 'A2'
wb.save(OUTPUT)
print(f'\n✅ Fichier enregistré : {OUTPUT}')
print(f'   {len(merged)} lignes  |  {len(HEADERS)} colonnes  |  1 seule feuille')

📋 81 feuilles trouvées : ['coordonne', '2026-01-13', '2026-01-14', '2026-01-15', '2026-01-16', '2026-01-17', '2026-01-18', '2026-01-19', '2026-01-20', '2026-01-21', '2026-01-22', '2026-01-23', '2026-01-24', '2026-01-25', '2026-01-26', '2026-01-27', '2026-01-28', '2026-01-29', '2026-01-30', '2026-01-31', '2026-02-01', '2026-02-02', '2026-02-03', '2026-02-04', '2026-02-05', '2026-02-06', '2026-02-07', '2026-02-08', '2026-02-09', '2026-02-10', '2026-02-11', '2026-02-12', '2026-02-13', '2026-02-14', '2026-02-15', '2026-02-16', '2026-02-17', '2026-02-18', '2026-02-19', '2026-02-20', '2026-02-21', '2026-02-22', '2026-02-23', '2026-02-24', '2026-02-25', '2026-02-26', '2026-02-27', '2026-02-28', '2026-03-01', '2026-03-02', '2026-03-03', '2026-03-04', '2026-03-05', '2026-03-06', '2026-03-07', '2026-03-08', '2026-03-09', '2026-03-10', '2026-03-11', '2026-03-12', '2026-03-13', '2026-03-14', '2026-03-15', '2026-03-16', '2026-03-17', '2026-03-18', '2026-03-19', '2026-03-20', '2026-03-21', '2026-03-

Cette section est le cœur du traitement des données. Elle effectue les étapes suivantes :

1.  **Paramètres :** Définit des constantes comme le nom de la feuille de coordonnées GPS, les valeurs pour 'Variete' et 'Site', le nombre de points par répétition de coordonnées, et le nom du fichier de sortie.
2.  **Lecture des feuilles :** Lit toutes les feuilles du fichier Excel source dans un dictionnaire de DataFrames pandas.
3.  **Extraction des coordonnées GPS :** Isole la feuille 'Coordonnée' et extrait les informations de latitude, longitude, élévation et nom de point GPS pour chaque point.
4.  **Fusion des données :** Parcourt chaque feuille de données (hors 'Coordonnée'), puis chaque ligne de ces feuilles. Pour chaque ligne, il associe les données mesurées aux coordonnées GPS correspondantes (répétées tous les `NB_POINTS`) et ajoute les colonnes 'Variete' et 'Site'.
5.  **Création et formatage du fichier Excel de sortie :**
    *   Définit les en-têtes de colonne et leurs largeurs.
    *   Crée un nouveau classeur et une nouvelle feuille Excel.
    *   Écrit les en-têtes avec un formatage spécifique (gras, centré, sans couleur de fond).
    *   Écrit toutes les lignes de données fusionnées, en alternant les couleurs de fond (blanc et bleu clair) par groupes de `NB_POINTS` pour améliorer la lisibilité.
    *   Ajuste la largeur des colonnes et fige les volets pour que l'en-tête reste visible lors du défilement.

In [ ]:
from google.colab import files
files.download(OUTPUT)
print('📥 Téléchargement lancé !')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Téléchargement lancé !


In [ ]:
display(df_verified.head())

,Date,SN,Description,Time,Temp (℃),Humidity (%),EC (μS/cm),PH,Nitrogen (mg/kg),Phosphorus (mg/kg),Potassium (mg/kg),Salinity (mg/kg),Point GPS,Latitude,Longitude,Elevation (m),Variete,site
0,13-01-2026,1,FG1,2026-01-13 12:50:50,26.6,45.1,440,8.0,23,31,71,242,F1FNY,16.532246,-15.198346,27.898767,fanaye1,fanaye
1,13-01-2026,2,FG2,2026-01-13 12:51:13,25.1,38.9,213,7.8,13,17,37,117,F2FNY,16.532222,-15.198356,27.898767,fanaye1,fanaye
2,13-01-2026,3,FG3,2026-01-13 12:51:33,24.4,38.4,221,7.7,12,16,36,122,F3FNY,16.532194,-15.198367,27.898767,fanaye1,fanaye
3,13-01-2026,4,FG4,2026-01-13 12:51:48,23.7,32.0,240,8.0,13,17,39,132,F4FNY,16.532167,-15.198383,27.898767,fanaye1,fanaye
4,13-01-2026,5,FG5,2026-01-13 12:52:18,21.8,33.1,246,8.1,15,20,42,135,F5FNY,16.532121,-15.198305,27.898767,fanaye1,fanaye


Cette cellule lance le téléchargement du fichier Excel final (`fanaye1_grand_parc_fanaye.xlsx`) qui contient toutes les données fusionnées et formatées. Le message '📥 Téléchargement lancé !' confirme que l'opération a été initiée.

## Étape 4 — Téléchargement

In [ ]:
from google.colab import files
files.download(OUTPUT)
print('📥 Téléchargement lancé !')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Téléchargement lancé !


In [ ]:
import pandas as pd

# Charger le fichier Excel dans un DataFrame pandas
df_verified = pd.read_excel(OUTPUT)

print('Aperçu des premières lignes du fichier Excel fusionné :')
display(df_verified.head())

print('\nInformations sur le DataFrame :')
df_verified.info()

Aperçu des premières lignes du fichier Excel fusionné :


,Date,SN,Description,Time,Temp (℃),Humidity (%),EC (μS/cm),PH,Nitrogen (mg/kg),Phosphorus (mg/kg),Potassium (mg/kg),Salinity (mg/kg),Point GPS,Latitude,Longitude,Elevation (m),Variete,site
0,2026-01-13,1,AL1,2026-01-13 13:08:50,24.3℃,44.2%,200μS/cm,7.0,10mg/kg,14mg/kg,32mg/kg,110mg/kg,F1,16.532347,-15.198294,0,alioune1,alioune
1,2026-01-13,2,AL2,2026-01-13 13:09:10,23.6℃,57.7%,209μS/cm,7.6,11mg/kg,15mg/kg,34mg/kg,115mg/kg,F2,16.532317,-15.198300,0,alioune1,alioune
2,2026-01-13,3,AL3,2026-01-13 13:09:28,23.9℃,35.8%,233μS/cm,7.4,12mg/kg,17mg/kg,38mg/kg,128mg/kg,F3,16.532292,-15.198310,0,alioune1,alioune
3,2026-01-13,4,AL4,2026-01-13 13:09:49,23.8℃,33.8%,313μS/cm,7.4,16mg/kg,22mg/kg,51mg/kg,172mg/kg,F4,16.532263,-15.198323,0,alioune1,alioune
4,2026-01-13,5,AL5,2026-01-13 13:10:19,22.8℃,34.8%,240μS/cm,7.3,12mg/kg,16mg/kg,38mg/kg,132mg/kg,F5,16.532229,-15.198242,0,alioune1,alioune



Informations sur le DataFrame :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1922 entries, 0 to 1921
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                1922 non-null   object 
 1   SN                  1922 non-null   int64  
 2   Description         1922 non-null   object 
 3   Time                1922 non-null   object 
 4   Temp (℃)            1922 non-null   object 
 5   Humidity (%)        1922 non-null   object 
 6   EC (μS/cm)          1922 non-null   object 
 7   PH                  1922 non-null   float64
 8   Nitrogen (mg/kg)    1922 non-null   object 
 9   Phosphorus (mg/kg)  1922 non-null   object 
 10  Potassium (mg/kg)   1922 non-null   object 
 11  Salinity (mg/kg)    1922 non-null   object 
 12  Point GPS           1922 non-null   object 
 13  Latitude            1922 non-null   float64
 14  Longitude           1922 non-null   float64
 15  Elevation (m)       19

Cette section vérifie le contenu du fichier Excel fusionné. Elle charge le fichier dans un DataFrame pandas (`df_verified`), affiche les cinq premières lignes (`df_verified.head()`) pour un aperçu rapide et fournit des informations détaillées sur les colonnes (`df_verified.info()`), y compris leur nom, le nombre de valeurs non nulles et leur type de données. Cela permet de s'assurer que le fichier a été correctement créé et que les données sont structurées comme prévu.

### Nettoyage des colonnes avec unités
Nous allons supprimer les unités des colonnes `Temp (℃)`, `Humidity (%)`, `EC (μS/cm)`, `Nitrogen (mg/kg)`, `Phosphorus (mg/kg)`, `Potassium (mg/kg)`, `Salinity (mg/kg)` et les convertir en type numérique.

In [ ]:
# Fonction pour nettoyer et convertir les colonnes
def clean_and_convert(df, column_name, unit_to_remove):
    if column_name in df.columns:
        # Convertir en string d'abord pour gérer les erreurs et les types mixtes
        df[column_name] = df[column_name].astype(str)
        # Remplacer les virgules par des points pour la conversion numérique
        df[column_name] = df[column_name].str.replace(',', '.', regex=False)
        # Supprimer l'unité de la colonne
        df[column_name] = df[column_name].str.replace(unit_to_remove, '', regex=False).str.strip()
        # Convertir en numérique, avec des NaN pour les erreurs de conversion
        df[column_name] = pd.to_numeric(df[column_name], errors='coerce')
        print(f'Colonne "{column_name}" nettoyée et convertie en numérique.')
    else:
        print(f'La colonne "{column_name}" n\'existe pas dans le DataFrame.')


# Appliquer le nettoyage aux colonnes concernées
clean_and_convert(df_verified, 'Temp (℃)', '℃')
clean_and_convert(df_verified, 'Humidity (%)', '%')
clean_and_convert(df_verified, 'EC (μS/cm)', 'μS/cm')
clean_and_convert(df_verified, 'Nitrogen (mg/kg)', 'mg/kg')
clean_and_convert(df_verified, 'Phosphorus (mg/kg)', 'mg/kg')
clean_and_convert(df_verified, 'Potassium (mg/kg)', 'mg/kg')
clean_and_convert(df_verified, 'Salinity (mg/kg)', 'mg/kg')

# Afficher les informations mises à jour sur le DataFrame
print('\nInformations sur le DataFrame après nettoyage :')
df_verified.info()

print('\nAperçu des premières lignes après nettoyage :')
display(df_verified.head())

Colonne "Temp (℃)" nettoyée et convertie en numérique.
Colonne "Humidity (%)" nettoyée et convertie en numérique.
Colonne "EC (μS/cm)" nettoyée et convertie en numérique.
Colonne "Nitrogen (mg/kg)" nettoyée et convertie en numérique.
Colonne "Phosphorus (mg/kg)" nettoyée et convertie en numérique.
Colonne "Potassium (mg/kg)" nettoyée et convertie en numérique.
Colonne "Salinity (mg/kg)" nettoyée et convertie en numérique.

Informations sur le DataFrame après nettoyage :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1922 entries, 0 to 1921
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                1922 non-null   object 
 1   SN                  1922 non-null   int64  
 2   Description         1922 non-null   object 
 3   Time                1922 non-null   object 
 4   Temp (℃)            1922 non-null   float64
 5   Humidity (%)        1922 non-null   float64
 6   EC (μS/cm)          1

,Date,SN,Description,Time,Temp (℃),Humidity (%),EC (μS/cm),PH,Nitrogen (mg/kg),Phosphorus (mg/kg),Potassium (mg/kg),Salinity (mg/kg),Point GPS,Latitude,Longitude,Elevation (m),Variete,site
0,2026-01-13,1,AL1,2026-01-13 13:08:50,24.3,44.2,200,7.0,10,14,32,110,F1,16.532347,-15.198294,0,alioune1,alioune
1,2026-01-13,2,AL2,2026-01-13 13:09:10,23.6,57.7,209,7.6,11,15,34,115,F2,16.532317,-15.198300,0,alioune1,alioune
2,2026-01-13,3,AL3,2026-01-13 13:09:28,23.9,35.8,233,7.4,12,17,38,128,F3,16.532292,-15.198310,0,alioune1,alioune
3,2026-01-13,4,AL4,2026-01-13 13:09:49,23.8,33.8,313,7.4,16,22,51,172,F4,16.532263,-15.198323,0,alioune1,alioune
4,2026-01-13,5,AL5,2026-01-13 13:10:19,22.8,34.8,240,7.3,12,16,38,132,F5,16.532229,-15.198242,0,alioune1,alioune


Après la vérification initiale, cette section se concentre sur le nettoyage des données. Elle définit une fonction `clean_and_convert` qui:

1.  Convertit le contenu de la colonne en `string`.
2.  Remplace les virgules (`,`) par des points (`.`) pour permettre une conversion numérique correcte.
3.  Supprime les unités (`℃`, `%`, `μS/cm`, `mg/kg`) des valeurs.
4.  Convertit les colonnes en type numérique (float ou int), en gérant les erreurs de conversion par `NaN`.

Cette fonction est ensuite appliquée à plusieurs colonnes (`Temp (℃)`, `Humidity (%)`, `EC (μS/cm)`, `Nitrogen (mg/kg)`, `Phosphorus (mg/kg)`, `Potassium (mg/kg)`, `Salinity (mg/kg)`) pour nettoyer les données et les rendre utilisables pour des analyses quantitatives. Enfin, `df_verified.info()` et `df_verified.head()` sont à nouveau affichés pour confirmer que les types de données ont été mis à jour et que les premières lignes sont correctement nettoyées.